# Fake News Detection with Hybrid BERT + BiLSTM

This notebook implements a Fake News Detection model using the WELFake dataset.
It covers:
1. Data Loading & EDA
2. Preprocessing (Fill-and-Merge Strategy)
3. Model Implementation (BERT + BiLSTM)
4. Training & Evaluation

In [ ]:
# Install dependencies
!pip install transformers torch pandas scikit-learn matplotlib seaborn wordcloud

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import re
import string

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Data Loading and EDA

In [ ]:
# Load dataset
df = pd.read_csv('data/WELFake_Dataset.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Strategy: Fill-and-Merge
# 1. Fill NaN with empty string
df.fillna('', inplace=True)

# 2. Merge Title and Text with [SEP]
df['combined_text'] = df['title'] + ' [SEP] ' + df['text']

print(f'Shape after filling nulls: {df.shape}')
print('Example combined text:')
print(df['combined_text'].iloc[0][:200])

In [ ]:
# Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='label', data=df)
plt.title('Class Distribution (0=Fake, 1=Real)')
plt.show()

In [ ]:
# Text Length Distribution (using combined text)
df['text_len'] = df['combined_text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 6))
sns.histplot(df['text_len'], bins=50, kde=True)
plt.title('Text Length Distribution (Combined Text)')
plt.xlabel('Number of Words')
plt.xlim(0, 2000) # Limit x-axis for better visibility
plt.show()

In [ ]:
# Word Cloud for Fake News
fake_text = ' '.join(df[df['label'] == 0]['combined_text'].astype(str).tolist())
wordcloud_fake = WordCloud(width=800, height=400, background_color='black').generate(fake_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_fake, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Fake News')

In [ ]:
# Word Cloud for Real News
real_text = ' '.join(df[df['label'] == 1]['combined_text'].astype(str).tolist())
wordcloud_real = WordCloud(width=800, height=400, background_color='white').generate(real_text)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_real, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Real News')

## 2. Preprocessing

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

# Apply cleaning to combined text
df['clean_text'] = df['combined_text'].apply(clean_text)

In [ ]:
# Split Data
X_train, X_val, y_train, y_val = train_test_split(df['clean_text'], df['label'], test_size=0.2, random_state=42)
print(f'Train size: {len(X_train)}, Validation size: {len(X_val)}')

In [ ]:
# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN = 512
BATCH_SIZE = 16

In [ ]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
def create_data_loader(df_text, df_label, tokenizer, max_len, batch_size):
    ds = FakeNewsDataset(
        texts=df_text.to_numpy(),
        labels=df_label.to_numpy(),
        tokenizer=tokenizer,
        max_len=max_len
    )

    return DataLoader(
        ds,
        batch_size=batch_size,
        num_workers=2
    )

train_data_loader = create_data_loader(X_train, y_train, tokenizer, MAX_LEN, BATCH_SIZE)
val_data_loader = create_data_loader(X_val, y_val, tokenizer, MAX_LEN, BATCH_SIZE)

## 3. Model Architecture (Hybrid BERT + BiLSTM)

In [ ]:
class HybridBertBiLSTM(nn.Module):
    def __init__(self, n_classes):
        super(HybridBertBiLSTM, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.lstm = nn.LSTM(input_size=768, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(p=0.3)
        self.out = nn.Linear(128 * 2, n_classes) # *2 for bidirectional

    def forward(self, input_ids, attention_mask):
        # BERT Output
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # We use the last_hidden_state for LSTM
        last_hidden_state = outputs.last_hidden_state # (batch_size, seq_len, 768)
        
        # LSTM
        lstm_out, _ = self.lstm(last_hidden_state)
        
        # We can take the output of the last time step, or pool it. 
        # Here we take the mean of the LSTM outputs (Global Average Pooling) or just the last state.
        # Let's use the representation of the [CLS] token equivalent from LSTM or just mean pooling.
        # Simple approach: Mean pooling over sequence dimension
        # lstm_out shape: (batch_size, seq_len, hidden_size*2)
        
        # Alternatively, take the last time step output
        # out = lstm_out[:, -1, :]
        
        # Let's try mean pooling for better context capture
        out = torch.mean(lstm_out, dim=1)
        
        out = self.drop(out)
        return self.out(out)

In [ ]:
model = HybridBertBiLSTM(n_classes=2)
model = model.to(device)

## 4. Training

In [ ]:
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
total_steps = len(train_data_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss().to(device)

In [ ]:
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0

    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        _, preds = torch.max(outputs, dim=1)
        loss = loss_fn(outputs, targets)

        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / n_examples, np.mean(losses)

In [ ]:
def eval_model(model, data_loader, loss_fn, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0

    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            _, preds = torch.max(outputs, dim=1)
            loss = loss_fn(outputs, targets)

            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())

    return correct_predictions.double() / n_examples, np.mean(losses)

In [ ]:
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}
best_accuracy = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 10)

    train_acc, train_loss = train_epoch(
        model,
        train_data_loader,
        loss_fn,
        optimizer,
        device,
        scheduler,
        len(X_train)
    )

    print(f'Train loss {train_loss} accuracy {train_acc}')

    val_acc, val_loss = eval_model(
        model,
        val_data_loader,
        loss_fn,
        device,
        len(X_val)
    )

    print(f'Val   loss {val_loss} accuracy {val_acc}')
    print()

    history['train_acc'].append(train_acc.item())
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc.item())
    history['val_loss'].append(val_loss)

    if val_acc > best_accuracy:
        torch.save(model.state_dict(), 'best_model_state.bin')
        best_accuracy = val_acc

## 5. Evaluation & Visualization

In [ ]:
# Plot Training History
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='train accuracy')
plt.plot(history['val_acc'], label='validation accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='train loss')
plt.plot(history['val_loss'], label='validation loss')
plt.title('Loss over Epochs')
plt.legend()

plt.show()

In [ ]:
# Confusion Matrix & Classification Report
def get_predictions(model, data_loader):
    model = model.eval()
    predictions = []
    prediction_probs = []
    real_values = []

    with torch.no_grad():
        for d in data_loader:
            texts = d['text']
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            _, preds = torch.max(outputs, dim=1)

            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)

    predictions = torch.stack(predictions).cpu()
    prediction_probs = torch.stack(prediction_probs).cpu()
    real_values = torch.stack(real_values).cpu()
    return predictions, prediction_probs, real_values

y_pred, y_pred_probs, y_test = get_predictions(
    model,
    val_data_loader
)

print(classification_report(y_test, y_pred, target_names=['Fake', 'Real']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
df_cm = pd.DataFrame(cm, index=['Fake', 'Real'], columns=['Fake', 'Real'])
plt.figure(figsize=(6, 4))
sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()